<a href="https://colab.research.google.com/github/M4rck0/Procesamiento_Clasificacion_Datos/blob/main/Tarea_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Librerías
import kagglehub
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Modelos de clasificación
from sklearn.svm import LinearSVC # Separa con línea, plano, hiperplano
from sklearn.naive_bayes import MultinomialNB # Probabilidades
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Funciones

# Clasificación
def clasificar_sentimiento(rating):
    if rating <= 2:
        return "negativo"
    elif rating == 3:
        return "neutral"
    else:
        return "positivo"

In [3]:
# Descargar Kaggle
ruta = kagglehub.dataset_download("dongrelaxman/amazon-reviews-dataset")

100%|██████████| 4.59M/4.59M [00:00<00:00, 125MB/s]

Extracting files...


In [4]:
# Archivo
os.listdir(ruta)

['Amazon_Reviews.csv']

In [5]:
# Leer base
ruta_archivo = os.path.join(ruta, "Amazon_Reviews.csv")
df_amazon = pd.read_csv(ruta_archivo, engine = "python")

In [ ]:
df_amazon.to_csv('df_amazon.csv', index = False)

In [ ]:
os.getcwd()

'/content'

In [6]:
# Selección columnas
df_amazon = df_amazon[["Country", "Review Count", "Review Date", "Rating", "Review Title", "Review Text"]].copy()

# Eliminar filas sin reseña
df_amazon = df_amazon.dropna(subset=["Rating", "Review Text"])

# Extraer número de estrellas (rating)
df_amazon["rating"] = df_amazon["Rating"].str.extract(r"Rated (\d+) out of 5 stars").astype(float)

# Eliminar ratings que no se pudieron convertir
df_amazon = df_amazon.dropna(subset=["rating"])

df_amazon.head()

,Country,Review Count,Review Date,Rating,Review Title,Review Text,rating
0,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...",1.0
1,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,1.0
2,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,1.0
3,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,1.0
4,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,1.0


In [7]:
df_amazon["sentimiento"] = df_amazon["rating"].apply(clasificar_sentimiento)

df_amazon[["Review Text", "rating", "sentimiento"]].head()

,Review Text,rating,sentimiento
0,"I registered on the website, tried to order a ...",1.0,negativo
1,Had multiple orders one turned up and driver h...,1.0,negativo
2,I informed these reprobates that I WOULD NOT B...,1.0,negativo
3,I have bought from Amazon before and no proble...,1.0,negativo
4,If I could give a lower rate I would! I cancel...,1.0,negativo


In [8]:
df_amazon["sentimiento"].value_counts()

,count
sentimiento,
negativo,14350
positivo,5820
neutral,885


In [9]:
df_amazon["sentimiento"].value_counts(normalize=True)

,proportion
sentimiento,
negativo,0.681548
positivo,0.276419
neutral,0.042033


In [10]:
df_amazon["rating"].value_counts().sort_index()

,count
rating,
1.0,13123
2.0,1227
3.0,885
4.0,1292
5.0,4528


In [11]:
# Columna texto
df_amazon["texto"] = (df_amazon["Review Title"].fillna("") + " " + df_amazon["Review Text"].fillna(""))
df_amazon[["texto", "rating", "sentimiento"]].head()

,texto,rating,sentimiento
0,A Store That Doesn't Want to Sell Anything I r...,1.0,negativo
1,Had multiple orders one turned up and… Had mul...,1.0,negativo
2,I informed these reprobates I informed these r...,1.0,negativo
3,Advertise one price then increase it on websit...,1.0,negativo
4,If I could give a lower rate I would If I coul...,1.0,negativo


In [12]:
vectorizador = TfidfVectorizer(lowercase=True, stop_words="english", max_features=5000) # Mínusculas, inglés y máximo 5,000 palabras

X = vectorizador.fit_transform(df_amazon["texto"])
y = df_amazon["sentimiento"]

print(X.shape)

(21055, 5000)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Proporciones similares

# Diccionario de modelos a comparar
modelos = {
    "Regresión Logística": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Linear SVC": LinearSVC(
        class_weight="balanced",
        random_state=42
    ),

    "Multinomial Naive Bayes": MultinomialNB(),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

In [17]:
resultados = []

for nombre_modelo, modelo in modelos.items():

    # Entrenar modelo
    modelo.fit(X_train, y_train)

    # Predecir
    y_pred = modelo.predict(X_test)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    # Resultados
    resultados.append({
        "modelo": nombre_modelo,
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

    print(nombre_modelo)
    print(classification_report(y_test, y_pred))
    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))

Regresión Logística
              precision    recall  f1-score   support

    negativo       0.95      0.89      0.92      2870
     neutral       0.17      0.40      0.23       177
    positivo       0.88      0.85      0.87      1164

    accuracy                           0.85      4211
   macro avg       0.67      0.71      0.67      4211
weighted avg       0.90      0.85      0.87      4211

Matriz de confusión:
[[2541  242   87]
 [  64   70   43]
 [  63  112  989]]
Linear SVC
              precision    recall  f1-score   support

    negativo       0.94      0.94      0.94      2870
     neutral       0.20      0.20      0.20       177
    positivo       0.87      0.88      0.88      1164

    accuracy                           0.89      4211
   macro avg       0.67      0.67      0.67      4211
weighted avg       0.89      0.89      0.89      4211

Matriz de confusión:
[[2685   92   93]
 [  88   35   54]
 [  92   49 1023]]
Multinomial Naive Bayes
              precision    reca

In [18]:
# Resultados
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by="f1_macro", ascending=False)
df_resultados

,modelo,accuracy,f1_macro,f1_weighted
0,Regresión Logística,0.854904,0.672336,0.874711
1,Linear SVC,0.888863,0.670421,0.888817
3,Random Forest,0.853004,0.612549,0.845122
2,Multinomial Naive Bayes,0.891000,0.594486,0.870511


## Fórmulas de métricas de clasificación

### Accuracy

$$
Accuracy = \frac{\text{Predicciones correctas}}{\text{Total de observaciones}}
$$

También puede expresarse como:

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

### Precision

$$
Precision = \frac{TP}{TP + FP}
$$

### Recall

$$
Recall = \frac{TP}{TP + FN}
$$

### F1-score

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

### Support

$$
Support = \text{Número real de observaciones de cada clase}
$$

### Macro avg

El promedio macro calcula el promedio simple de una métrica entre todas las clases.

$$
Precision_{macro} = \frac{Precision_{negativo} + Precision_{neutral} + Precision_{positivo}}{3}
$$

$$
Recall_{macro} = \frac{Recall_{negativo} + Recall_{neutral} + Recall_{positivo}}{3}
$$

$$
F1_{macro} = \frac{F1_{negativo} + F1_{neutral} + F1_{positivo}}{3}
$$

### Weighted avg

El promedio ponderado calcula el promedio de una métrica considerando el número de observaciones de cada clase.

$$
Métrica_{weighted} =
\frac{
\sum_{i=1}^{k} Métrica_i \cdot Support_i
}{
\sum_{i=1}^{k} Support_i
}
$$

Para el caso específico de F1-score:

$$
F1_{weighted} =
\frac{
F1_{negativo} \cdot Support_{negativo} +
F1_{neutral} \cdot Support_{neutral} +
F1_{positivo} \cdot Support_{positivo}
}{
Support_{negativo} + Support_{neutral} + Support_{positivo}
}
$$

Se compararon cuatro modelos de clasificación de texto. Aunque Multinomial Naive Bayes obtuvo la mayor exactitud, no clasificó correctamente las reseñas neutrales. Por ello, la Regresión Logística fue el mejor modelo general, ya que obtuvo el mayor F1 macro y tuvo un desempeño más equilibrado entre clases. El principal reto identificado fue el desbalance de datos, especialmente en la clase neutral.